# 02 歷史目的地候選排序
以 scikit-learn `HistGradientBoostingClassifier` 對候選目的地作二元評分，再按分數排序。輸入是快照前歷史，不使用本次位置或出行時間。分數未校準，不能解讀為真實出行機率。
真實競賽資料與逐筆結果未包含於公開版本；作者同意公開的彙總命中率列於文末。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
CSV_PATH = Path('../data/yoxi_cleaned.csv')
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'找不到清理檔：{CSV_PATH}；請將已授權的清理檔放在專案 data/ 目錄。')
NEEDED = ['Id', 'RiderId', 'OrderFormCreatedTime_UTC8', 'DropOffLatitude', 'DropOffLongitude']
trips = pd.read_csv(CSV_PATH, encoding='utf-8-sig', usecols=NEEDED, dtype={'Id': 'string',
    'RiderId': 'string'}, low_memory=False)
trips['OrderFormCreatedTime_UTC8'] = pd.to_datetime(trips['OrderFormCreatedTime_UTC8'], errors='coerce')
for c in ('DropOffLatitude', 'DropOffLongitude'):
    trips[c] = pd.to_numeric(trips[c], errors='coerce')
if trips[NEEDED].isna().any().any():
    raise ValueError('預測必要欄位有缺值；請先回到 cleaning 檢查，02 保留原資料，請先核對輸入。')
if trips['Id'].duplicated().any():
    raise ValueError('Id 有重複，請先核對 01 的清理檔。')
CELL_M = 500
LAT_METERS_PER_DEG = 111320.0
LON_METERS_PER_DEG = 111320.0 * np.cos(np.deg2rad(24.5))

def make_cell(lat, lon, cell_m=CELL_M):
    x = np.floor(pd.to_numeric(lon, errors='coerce') * LON_METERS_PER_DEG / cell_m)
    y = np.floor(pd.to_numeric(lat, errors='coerce') * LAT_METERS_PER_DEG / cell_m)
    return 'g' + str(cell_m) + '_' + x.astype('Int64').astype('string') + '_' + y.astype('Int64').astype('string')
trips['dropoff_area'] = make_cell(trips['DropOffLatitude'], trips['DropOffLongitude'])
if trips['dropoff_area'].isna().any():
    raise ValueError('目的區域建立失敗。')
trips = trips.sort_values(['OrderFormCreatedTime_UTC8', 'Id'], kind='stable').reset_index(drop=True)
T = 'OrderFormCreatedTime_UTC8'
print('讀取行程／欄位：', f'{len(trips):,}', '/', len(NEEDED), '；下車網格：', trips['dropoff_area'].nunique())
print('訂單時間：', trips[T].min(), '～', trips[T].max())


## 時間快照
快照以訂單建立時間切分，歷史皆嚴格早於 cutoff。每段答案是該期間每帳號第一趟，且快照前至少 3 趟。

| 快照 | cutoff（含於答案期間） | 答案期間結束（不含） |
| --- | --- | --- |
| train_a | 2026-02-15 | 2026-03-01 |
| train_b | 2026-03-01 | 2026-03-16 |
| validation | 2026-03-16 | 2026-04-01 |
| april | 2026-04-01 | 2026-05-01 |

同一帳號可出現在不同快照，但候選與特徵每次只由該快照之前資料建立。網格沿用 01 的固定近似 500 公尺換算與邊界限制。

In [ ]:
WINDOWS = [('train_a', '2026-02-15', '2026-03-01'), ('train_b', '2026-03-01', '2026-03-16'), ('validation',
    '2026-03-16', '2026-04-01'), ('april', '2026-04-01', '2026-05-01')]

def historical_snapshot(all_trips, cutoff, end):
    cutoff, end = (pd.Timestamp(cutoff), pd.Timestamp(end))
    history = all_trips.loc[all_trips[T].lt(cutoff)]
    future = all_trips.loc[all_trips[T].ge(cutoff) & all_trips[T].lt(end)]
    first = future.sort_values(['RiderId', T, 'Id'], kind='stable').drop_duplicates('RiderId',
        keep='first')[['RiderId', 'dropoff_area']].rename(columns={'dropoff_area': 'actual_area'})
    counts = history.groupby('RiderId', observed=True).size().rename('history_rides')
    answers = first.join(counts, on='RiderId')
    answers = answers.loc[answers['history_rides'].ge(3)].copy()
    if answers['RiderId'].duplicated().any():
        raise AssertionError('每個快照每位帳號應只有一個答案。')
    return (history, answers)


## 候選與特徵
候選為個人歷史目的地 Top-10 與全體熱門 Top-3 的聯集。個人依趟數排序；全體依不重複帳號數、趟數排序；同分以網格編號排序。各帳號最多 13 個候選。
特徵包含帳號趟數、活躍週數、最近搭乘距今天數，以及目的地頻率、占比、排名、最近搭乘距今天數和全體熱門程度。新目的地的個人次數補 0、未知排名或天數補 999。

In [ ]:
FEATURES = ['log_user_rides', 'active_weeks', 'user_recency_days', 'log_dest_rides', 'personal_share',
    'personal_rank', 'dest_recency_days', 'log_global_riders', 'log_global_rides', 'is_personal_top10',
    'is_global_top3']

def build_candidates(history, answers, cutoff):
    cutoff = pd.Timestamp(cutoff)
    if answers.empty:
        raise ValueError(f'{cutoff.date()} 沒有符合資格的評估帳號。')
    eligible = answers['RiderId']
    personal_h = history.loc[history['RiderId'].isin(eligible), ['RiderId', 'dropoff_area', T]]
    user = personal_h.groupby('RiderId', observed=True).agg(user_rides=('dropoff_area', 'size'),
        user_last=(T, 'max'))
    user['active_weeks'] = personal_h.assign(week=personal_h[T].dt.to_period('W')).groupby('RiderId',
        observed=True)['week'].nunique()
    user['log_user_rides'] = np.log1p(user['user_rides'])
    user['user_recency_days'] = ((cutoff - user['user_last']).dt.total_seconds() / 86400).clip(lower=0)
    user = user[['user_rides', 'active_weeks', 'log_user_rides', 'user_recency_days']].reset_index()
    personal = personal_h.groupby(['RiderId', 'dropoff_area'],
        observed=True).agg(dest_rides=('dropoff_area', 'size'), dest_last=(T, 'max')).reset_index()
    personal = personal.sort_values(['RiderId', 'dest_rides', 'dropoff_area'], ascending=[True, False,
        True], kind='stable')
    personal['personal_rank'] = personal.groupby('RiderId', observed=True).cumcount() + 1
    personal['is_personal_top10'] = personal['personal_rank'].le(10).astype('int8')
    personal_top = personal.loc[personal['is_personal_top10'].eq(1), ['RiderId', 'dropoff_area']]
    global_stats = history.groupby('dropoff_area', observed=True).agg(global_riders=('RiderId', 'nunique'),
        global_rides=('RiderId', 'size')).reset_index().sort_values(['global_riders', 'global_rides',
        'dropoff_area'], ascending=[False, False, True], kind='stable')
    global_stats['global_rank'] = np.arange(1, len(global_stats) + 1)
    top_global = global_stats.head(3)['dropoff_area'].tolist()
    global_candidates = pd.MultiIndex.from_product([eligible.tolist(), top_global], names=['RiderId',
        'dropoff_area']).to_frame(index=False)
    candidates = pd.concat([personal_top, global_candidates], ignore_index=True)
    candidates = candidates.drop_duplicates(['RiderId', 'dropoff_area'])
    candidates = candidates.merge(personal, on=['RiderId', 'dropoff_area'], how='left',
        validate='one_to_one')
    candidates = candidates.merge(global_stats, on='dropoff_area', how='left', validate='many_to_one')
    candidates = candidates.merge(user, on='RiderId', how='left', validate='many_to_one')
    candidates = candidates.merge(answers[['RiderId', 'actual_area']], on='RiderId', how='left',
        validate='many_to_one')
    candidates['dest_rides'] = candidates['dest_rides'].fillna(0)
    candidates['personal_rank'] = candidates['personal_rank'].fillna(999)
    candidates['is_personal_top10'] = candidates['is_personal_top10'].fillna(0)
    candidates['personal_share'] = candidates['dest_rides'] / candidates['user_rides']
    candidates['dest_recency_days'] = ((cutoff - candidates['dest_last']).dt.total_seconds() / 86400).clip(lower=0).fillna(999)
    candidates['log_dest_rides'] = np.log1p(candidates['dest_rides'])
    candidates['log_global_riders'] = np.log1p(candidates['global_riders'])
    candidates['log_global_rides'] = np.log1p(candidates['global_rides'])
    candidates['is_global_top3'] = candidates['global_rank'].le(3).astype('int8')
    candidates['y'] = candidates['dropoff_area'].eq(candidates['actual_area']).astype('int8')
    if candidates[FEATURES].isna().any().any() or candidates.duplicated(['RiderId', 'dropoff_area']).any():
        raise AssertionError('候選特徵缺值或同帳號同網格重複。')
    if candidates.groupby('RiderId', observed=True)['y'].sum().gt(1).any():
        raise AssertionError('每人每個快照最多一個正例。')
    for f in FEATURES:
        candidates[f] = candidates[f].astype('float32')
    return candidates
snapshots = {}
for name, cutoff, end in WINDOWS:
    h, a = historical_snapshot(trips, cutoff, end)
    snap = build_candidates(h, a, cutoff)
    snapshots[name] = snap
    print(f"{name}: {len(a):,} 位，{len(snap):,} 個候選；候選命中 {snap['y'].sum() / len(a):.2%}")
    del h, a


## 評估與選模
Top-1／Top-3 評估目的地網格命中。分母包含所有符合資格且答案期間有下一趟的帳號，包括候選沒有答案者；其命中記為 0。個人規則只用個人歷史候選，不用全體熱門補足。
此分母不代表全體乘客，也未評估開啟 App、叫車或通知轉換。候選涵蓋率是排序模型可達到的上限。

In [ ]:
def topk_metrics(frame, ranking, prefix):
    ranked = frame.sort_values(['RiderId', ranking, 'dropoff_area'], ascending=[True, False, True],
        kind='stable').copy()
    ranked['_rank'] = ranked.groupby('RiderId', observed=True).cumcount() + 1
    n = ranked['RiderId'].nunique()
    return {prefix + '_top1': ranked.loc[ranked['_rank'].eq(1), 'y'].sum() / n,
        prefix + '_top3': ranked.loc[ranked['_rank'].le(3)].groupby('RiderId',
        observed=True)['y'].max().sum() / n}

def evaluate(frame, scores=None):
    base = frame.copy()
    result = {'accounts': base['RiderId'].nunique(),
        'candidate_coverage': base['y'].sum() / base['RiderId'].nunique()}
    personal_only = base.loc[base['is_personal_top10'].eq(1)]
    if personal_only['RiderId'].nunique() != result['accounts']:
        raise AssertionError('每位合資格帳號都應有至少一個個人歷史目的網格。')
    result.update(topk_metrics(personal_only, 'dest_rides', 'personal'))
    if scores is not None:
        base['score'] = np.asarray(scores)
        result.update(topk_metrics(base, 'score', 'model'))
    return result
for name in ('validation', 'april'):
    print(name, evaluate(snapshots[name]))


兩組原有設定 A／B 在三月驗證集依 Top-3、Top-1、設定名稱順序選擇；平手偏向較簡單的 A。每個候選是一筆二元樣本，候選無答案的帳號仍保留其負例。

In [ ]:
train_frame = pd.concat([snapshots['train_a'], snapshots['train_b']], ignore_index=True)
valid_frame = snapshots['validation']
if train_frame['y'].nunique() < 2:
    raise ValueError('訓練資料只剩單一標籤，先檢查切分和候選覆蓋。')
CONFIGS = {'A': dict(max_iter=80, max_leaf_nodes=15, min_samples_leaf=40, learning_rate=0.06,
    l2_regularization=1.0), 'B': dict(max_iter=140, max_leaf_nodes=31, min_samples_leaf=40,
    learning_rate=0.06, l2_regularization=1.0)}
validation_rows = []
for name, params in CONFIGS.items():
    model = HistGradientBoostingClassifier(**params, early_stopping=False, random_state=42)
    model.fit(train_frame[FEATURES], train_frame['y'])
    scores = model.predict_proba(valid_frame[FEATURES])[:, 1]
    row = {'config': name, **evaluate(valid_frame, scores)}
    validation_rows.append(row)
    print('三月驗證：', name, 'Top-1', f"{row['model_top1']:.2%}", 'Top-3', f"{row['model_top3']:.2%}")
validation_table = pd.DataFrame(validation_rows).set_index('config')
display(validation_table)
CHOSEN = validation_table.reset_index().sort_values(['model_top3', 'model_top1', 'config'],
    ascending=[False, False, True], kind='stable').iloc[0]['config']
print('只依三月驗證選出：', CHOSEN)


## 四月回測與限制
選定設定後，合併 train_a、train_b、validation 的候選樣本重新訓練，再評估四月；四月答案不進入 fit。
四月已用於前期探索，不能視為未接觸的最終測試集。四月結果僅供固定流程的回測比較；若據此再選特徵或調參，需另設後續驗證期間。

In [ ]:
all_pre_april = pd.concat([train_frame, valid_frame], ignore_index=True)
final_model = HistGradientBoostingClassifier(**CONFIGS[CHOSEN], early_stopping=False, random_state=42)
final_model.fit(all_pre_april[FEATURES], all_pre_april['y'])
april_frame = snapshots['april']
april_scores = final_model.predict_proba(april_frame[FEATURES])[:, 1]
april_result = evaluate(april_frame, april_scores)
display(pd.DataFrame([april_result], index=['april']))
print('Top-3 相對個人頻率規則差距（百分點）：',
    f"{100 * (april_result['model_top3'] - april_result['personal_top3']):+.2f}")
print('符合評估帳號：', f"{april_result['accounts']:,}", '；候選命中率上限：', f"{april_result['candidate_coverage']:.2%}")


## 已記錄的評估結果

以下採作者既有紀錄中「修正口徑後的正式結果」。個人規則只使用個人歷史目的地，不以全體熱門地點補足 Top-3。本次未重新訓練或重算，未公開帳號數與逐筆預測。

| 評估期間 | 個人規則 Top-1 | 個人規則 Top-3 | ML B Top-1 | ML B Top-3 |
| --- | ---: | ---: | ---: | ---: |
| 三月後半驗證 | 32.14% | 48.45% | 33.01% | 48.97% |
| 四月首趟回測 | 29.76% | 45.77% | 30.47% | 46.29% |

三月驗證設定 A 的 Top-1／Top-3 為 32.90%／48.94%；依 Top-3 優先的既定選模規則採設定 B。四月 ML 相對個人規則，Top-1 約增加 0.71 個百分點、Top-3 約增加 0.52 個百分點，增益有限。

各期間分母包含符合歷史資格且該期間有下一趟的所有評估帳號，包含候選未涵蓋答案者。命中指目的地網格，不是地址或叫車機率。四月曾參與探索，這些結果是時間回測，不是未接觸的最終測試。
